# Building a custom detector

How to author a new geometry end-to-end: start from a preset, modify the YAML, load it, and run a simulation. See the [config schema](../../docs/detector/config-schema.md) for every field and [adding a detector](../../docs/contributing/adding-a-detector.md) for the frame invariants a new geometry must respect.

In [ ]:
import os, sys
_d = os.path.abspath(os.getcwd())
while _d != os.path.dirname(_d) and not os.path.isdir(os.path.join(_d, 'config')):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d); os.chdir(_d)

import yaml, tempfile, numpy as np, jax
import matplotlib.pyplot as plt
from tools.geometry import generate_detector
from tools.simulation import DetectorSimulator
from tools.loader import build_deposit_data
from tools.output import to_sparse
from tools.visualization import visualize_wire_signals

## Start from a preset and modify it

The cleanest way to author a config is to load a working preset and change what you need — here we make a **single-volume** wire TPC with a **coarser 0.5 cm wire pitch**. Each volume needs a `geometry` (ranges + drift_direction) and either `planes` (wire) or a pixel `readout`; the `simulation`/`readout`/`electric_field`/`medium` sections are shared.

In [ ]:
with open('config/cubic_wireplane_config.yaml') as f:
    base = yaml.safe_load(f)

cfgdict = dict(base)
cfgdict['volumes'] = [base['volumes'][0]]        # keep a single TPC volume
for p in cfgdict['volumes'][0]['planes']:
    p['wire_spacing'] = 0.5                       # coarser pitch than the 0.3 cm default

my_yaml = os.path.join(tempfile.gettempdir(), 'my_detector.yaml')
with open(my_yaml, 'w') as f:
    yaml.safe_dump(cfgdict, f)
print('wrote', my_yaml)

## Load it

`generate_detector` validates the YAML; `create_sim_config` (inside `DetectorSimulator`) derives wire counts, diffusion sigmas, and `num_time_steps`.

In [ ]:
detector = generate_detector(my_yaml)
sim = DetectorSimulator(detector, include_track_hits=False, include_digitize=True,
                        total_pad=10_000, response_chunk_size=5_000)
cfg = sim.config
sim.warm_up()
print('volumes:', len(cfg.volumes), ' planes/volume:', cfg.volumes[0].n_planes)

## Run a synthetic event

Same API as any other detector — the geometry is the only thing that changed.

In [ ]:
n = 300; s = np.arange(n) * 0.4
start = np.array([-100.0, 0.0, 0.0]); d = np.array([1.0, 0.3, 0.2]); d /= np.linalg.norm(d)
pts = np.clip(start[None, :] + s[:, None] * d[None, :], -215.9, 215.9)
pos_mm = (pts * 10).astype(np.float32)
de = np.full(n, 2.1 * 0.4, np.float32); dx = np.full(n, 0.4, np.float32); tid = np.zeros(n, np.int32)

deposits = build_deposit_data(pos_mm, de, dx, cfg, track_ids=tid)
signals, _, _ = sim.process_event(deposits, key=jax.random.PRNGKey(0))
sparse = to_sparse(signals, cfg, threshold_adc=1200 / cfg.electrons_per_adc)
visualize_wire_signals(sparse, cfg, threshold_enc=1200, gamma=0.3, sparse=True)
plt.show()

## Next steps

- [Config schema](../../docs/detector/config-schema.md) — every YAML field
- [Presets](../../docs/detector/presets.md) — the shipped detectors to start from
- [Adding a detector](../../docs/contributing/adding-a-detector.md) — frame & uniformity invariants
- [Capacities](../../docs/concepts/capacities.md) — size `total_pad`/`maxg` with the profiler